In [1]:
!pip install tfds
!pip install tfds-nightly
!pip install --upgrade tensorflow-datasets
!pip install tensorflow-model-optimization
!pip install tensorflow-model-optimization tensorflow-probability

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.2/79.2 kB 7.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.8/248.8 kB 24.3 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=92182c33530c274d91dbf297d1307eb18397e92b4a10c5a11726ae1a2b0cd688
  Stored in directory: /root/.cache/pip/wheels/1a/b0/8c/4b75c4116c31f83c8f9f047231251e13cc74481cca4a78a9ce
Successfully built docopt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 71.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 90.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Succes

In [1]:
import tensorflow_datasets as tfds
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np

from tensorflow.keras import layers, models
from tensorflow.keras.applications import (
    ResNet50,
    EfficientNetB0,
    resnet50,
    efficientnet
)
from tensorflow.keras.optimizers import Adam

# ---------------------------------------------------
# 1. Load the tf_flowers Dataset
# ---------------------------------------------------
# Create your own train/test split: 80% / 20%
(dataset_train, dataset_test), info = tfds.load(
    'oxford_flowers102',
    split=['train[:80%]', 'train[80%:]'],
    as_supervised=True,
    with_info=True
)


# Number of classes in tf_flowers
num_classes = info.features['label'].num_classes
nums = num_classes  # keep both variables as you did before

print(f"Number of classes: {num_classes}")

# ---------------------------------------------------
# 2. Convert to NumPy arrays (Optionally Resize)
# ---------------------------------------------------
IMG_SIZE = 128

x_train_list = []
y_train_list = []

for img, label in dataset_train:
    # Resize to reduce memory usage
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    x_train_list.append(img.numpy())
    y_train_list.append(label.numpy())

x_test_list = []
y_test_list = []

for img, label in dataset_test:
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    x_test_list.append(img.numpy())
    y_test_list.append(label.numpy())

# Convert lists to NumPy arrays
x_train = np.stack(x_train_list).astype("float32")
y_train = np.array(y_train_list)
x_test  = np.stack(x_test_list).astype("float32")
y_test  = np.array(y_test_list)

# ---------------------------------------------------
# 3. Normalize & One-Hot Encode Labels
# ---------------------------------------------------
# Normalize pixel values to [0, 1]
x_train /= 255.0
x_test  /= 255.0

# Convert integer labels to one-hot vectors
y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_test  = tf.keras.utils.to_categorical(y_test, num_classes)

print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_test  shape: {x_test.shape}")
print(f"y_test  shape: {y_test.shape}")
print(f"num_classes: {num_classes}, nums: {nums}")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

...

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_flowers102/incomplete.7XLYR4_2.1.1/oxford_flowers102-test.tfrecord-…

Generating validation examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_flowers102/incomplete.7XLYR4_2.1.1/oxford_flowers102-validation.tfr…

Dataset oxford_flowers102 downloaded and prepared to /root/tensorflow_datasets/oxford_flowers102/2.1.1. Subsequent calls will reuse this data.
Number of classes: 102
x_train shape: (816, 128, 128, 3)
y_train shape: (816, 102)
x_test  shape: (204, 128, 128, 3)
y_test  shape: (204, 102)
num_classes: 102, nums: 102


In [5]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Build on top of your already-loaded x_train, y_train, x_test, y_test

base_model = VGG16(weights='imagenet',
                   include_top=False,
                   input_shape=(128, 128, 3))

# Freeze convolutional base
for layer in base_model.layers:
    layer.trainable = False

# Add new classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
preds = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=preds)

model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# === Train for 1 epoch ===
history = model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=50,
    batch_size=32
)


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 128, 128, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 128, 128, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 64, 64, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 16, 16, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 8, 8, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 102)            │        52,326 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,029,670 (57.33 MB)

 Trainable params: 314,982 (1.20 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

Epoch 1/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 10s 272ms/step - accuracy: 0.0069 - loss: 4.8880 - val_accuracy: 0.0049 - val_loss: 4.6827
Epoch 2/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 73ms/step - accuracy: 0.0172 - loss: 4.7556 - val_accuracy: 0.0000e+00 - val_loss: 4.6500
Epoch 3/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 71ms/step - accuracy: 0.0200 - loss: 4.6964 - val_accuracy: 0.0098 - val_loss: 4.6277
Epoch 4/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 72ms/step - accuracy: 0.0259 - loss: 4.6044 - val_accuracy: 0.0049 - val_loss: 4.6101
Epoch 5/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 71ms/step - accuracy: 0.0185 - loss: 4.5915 - val_accuracy: 0.0147 - val_loss: 4.5949
Epoch 6/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 71ms/step - accuracy: 0.0326 - loss: 4.5282 - val_accuracy: 0.0245 - val_loss: 4.5782
Epoch 7/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - accuracy: 0.0281 - loss: 4.5194 - val_accuracy: 0.0245 - val_loss: 4.5587
Epoch 8/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 81ms/step - accuracy: 0.0446 - loss: 4.4806 - val_accuracy: 0.03

In [6]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Build on top of your already-loaded x_train, y_train, x_test, y_test

base_model = VGG16(weights='imagenet',
                   include_top=False,
                   input_shape=(128, 128, 3))

# Freeze convolutional base
for layer in base_model.layers:
    layer.trainable = False

# Add new classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
preds = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=preds)

model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# === Train for 1 epoch ===
history = model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=50,
    batch_size=32
)


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 128, 128, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 128, 128, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 64, 64, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 16, 16, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 8, 8, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 102)            │        52,326 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,029,670 (57.33 MB)

 Trainable params: 314,982 (1.20 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

Epoch 1/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 223ms/step - accuracy: 0.0111 - loss: 4.8790 - val_accuracy: 0.0147 - val_loss: 4.7071
Epoch 2/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - accuracy: 0.0218 - loss: 4.7265 - val_accuracy: 0.0147 - val_loss: 4.6726
Epoch 3/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 84ms/step - accuracy: 0.0054 - loss: 4.7493 - val_accuracy: 0.0196 - val_loss: 4.6468
Epoch 4/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 85ms/step - accuracy: 0.0158 - loss: 4.6734 - val_accuracy: 0.0147 - val_loss: 4.6264
Epoch 5/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 83ms/step - accuracy: 0.0177 - loss: 4.5628 - val_accuracy: 0.0147 - val_loss: 4.6072
Epoch 6/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step - accuracy: 0.0378 - loss: 4.5293 - val_accuracy: 0.0245 - val_loss: 4.5889
Epoch 7/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 74ms/step - accuracy: 0.0336 - loss: 4.5150 - val_accuracy: 0.0294 - val_loss: 4.5638
Epoch 8/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 74ms/step - accuracy: 0.0283 - loss: 4.4677 - val_accuracy: 0.0392 - 

In [17]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

import numpy as np
import networkx as nx

# ---------------------------------------------
# 1. Build Pretrained VGG-16 with Custom Head
# ---------------------------------------------
def build_vgg16_graph_head(input_shape=(128,128,3), num_classes=5):
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False

    x = Flatten(name="flatten")(base_model.output)
    x = Dense(512, activation='relu', name="graph_dense")(x)
    x = Dropout(0.5, name="dropout")(x)
    outputs = Dense(num_classes, activation='softmax', name="predictions")(x)

    model = Model(inputs=base_model.input, outputs=outputs)
    return model

model = build_vgg16_graph_head((128,128,3), num_classes)
model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

print(">>> Training original VGG-16 head for 1 epoch")
_ = model.fit(x_train, y_train, epochs=50, batch_size=32,
              validation_data=(x_test, y_test), verbose=1)

# ---------------------------------------------
# 2. Graph-Theory Pruning Function (same as yours)
# ---------------------------------------------
def graph_theoretic_prune_dense_layer(model, layer_name='graph_dense',
                                      prune_ratio=0.2, sim_threshold=0.5):

    dense_layer = model.get_layer(layer_name)
    kernel, bias = dense_layer.get_weights()         # (input_dim, units)
    input_dim, num_units = kernel.shape

    norms = np.linalg.norm(kernel, axis=0, keepdims=True) + 1e-8
    kernel_norm = kernel / norms
    similarity_matrix = np.dot(kernel_norm.T, kernel_norm)

    G = nx.Graph()
    G.add_nodes_from(range(num_units))

    for i in range(num_units):
        for j in range(i+1, num_units):
            sim = similarity_matrix[i, j]
            if sim > sim_threshold:
                G.add_edge(i, j, weight=sim)

    centrality = nx.eigenvector_centrality_numpy(G, weight='weight')
    sorted_units = sorted(centrality.items(), key=lambda x: x[1])
    num_prune = int(prune_ratio * num_units)
    pruned_units = [unit for unit, cent in sorted_units[:num_prune]]
    keep_units = [i for i in range(num_units) if i not in pruned_units]

    print(f"Pruning {num_prune}/{num_units} neurons from '{layer_name}'")

    new_kernel = kernel[:, keep_units]
    new_bias = bias[keep_units]

    # Rebuild new dense layer
    new_dense = Dense(len(keep_units), activation='relu', name=f'{layer_name}_pruned')
    new_dense.build((None, input_dim))
    new_dense.set_weights([new_kernel, new_bias])

    # Rebuild model: Input → flatten → new_dense → dropout → softmax
    base_input = model.input
    flatten_output = model.get_layer('flatten').output

    x = new_dense(flatten_output)
    x = model.get_layer('dropout')(x)
    out = model.get_layer('predictions')(x)

    new_model = Model(inputs=base_input, outputs=out)
    return new_model

# ---------------------------------------------
# 3. Apply Graph Pruning + Fine-Tune
# ---------------------------------------------
print("\n>>> Applying Graph-Pruning")
pruned_model = graph_theoretic_prune_dense_layer(model,
                                                layer_name='graph_dense',
                                                prune_ratio=0.5,
                                                sim_threshold=0.8)

pruned_model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

print("\n>>> Fine-tuning pruned model for 1 epoch")
history = pruned_model.fit(x_train, y_train, epochs=50, batch_size=32,
                           validation_data=(x_test, y_test), verbose=1)

# ---------------------------------------------
# 4. Evaluate
# ---------------------------------------------
loss, acc = pruned_model.evaluate(x_test, y_test, verbose=0)
print(f"Graph-Pruned VGG-16 Accuracy: {acc:.4f}")

# Save optional
# pruned_model.save("vgg16_graphpruned_flowers.h5")


>>> Training original VGG-16 head for 1 epoch
Epoch 1/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 10s 260ms/step - accuracy: 0.0077 - loss: 5.0003 - val_accuracy: 0.0245 - val_loss: 4.5626
Epoch 2/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 73ms/step - accuracy: 0.0562 - loss: 4.3786 - val_accuracy: 0.0735 - val_loss: 4.3813
Epoch 3/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 82ms/step - accuracy: 0.1222 - loss: 4.0265 - val_accuracy: 0.1275 - val_loss: 4.2432
Epoch 4/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 85ms/step - accuracy: 0.2372 - loss: 3.6774 - val_accuracy: 0.1618 - val_loss: 4.0464
Epoch 5/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 83ms/step - accuracy: 0.3080 - loss: 3.3448 - val_accuracy: 0.2108 - val_loss: 3.8638
Epoch 6/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 71ms/step - accuracy: 0.4100 - loss: 2.9604 - val_accuracy: 0.2549 - val_loss: 3.6715
Epoch 7/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 72ms/step - accuracy: 0.5040 - loss: 2.6000 - val_accuracy: 0.2941 - val_loss: 3.5234
Epoch 8/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 72ms/step - accuracy: 

AmbiguousSolution: `eigenvector_centrality_numpy` does not give consistent results for disconnected graphs